<a href="https://colab.research.google.com/github/aldo02032004/llm-topic-classification-pipeline/blob/main/topic_classification_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Topic Classification Pipeline: themes → top authors → stance

1. **Load data** from one or more Google Sheets / Drive exports.
2. **Remove media accounts**, so only organic accounts are ranked.
3. **Clean text**: strip RT/reply markers, turn emoji into words, drop short posts and duplicates.
4. **Assign themes**:
   - The LLM builds the themes and topic keywords from `TOPIC`.
   - Posts without any keyword go straight to `lainnya` (other).
   - The rest are labelled by the LLM.
5. **Top authors**: rank accounts per theme by post count, engagement and followers, and keep the top 10.
6. **Stance**: the LLM reads each top author's posts and returns a stance (positif / negatif / netral / kontroversi) with a short summary.

**Final output:** `kol_summary_<project>.csv` in Google Drive.

The LLM is **Gemma 2 9B SahabatAI** (GGUF, `q4_k_m`, ~5.8 GB), run with llama-cpp on the Colab GPU: select **Runtime → Change runtime type → T4 GPU**. No API key and no rate limit. Every answer is forced into a JSON schema, so it always parses.

## 1. Install

In [ ]:
%pip install -q "great @ git+https://github.com/azmkto/GREAT-Tools.git" emoji ftfy openpyxl tqdm pydantic huggingface-hub
# wheel CUDA siap pakai, tanpa compile
%pip install -q llama-cpp-python==0.3.36 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

## 2. Imports and local LLM

In [40]:
import json, os, re
from collections import Counter
from datetime import date
from typing import Annotated, Literal

import emoji
import pandas as pd
from google.colab import drive
from IPython.display import display
from llama_cpp import Llama
from pydantic import BaseModel, Field, create_model
from tqdm.auto import tqdm

from great import EXPORT_COLUMNS, clean_for_bert, clean_for_ner, is_media_account

tqdm.pandas()

LLM_MODEL_REPO = "faizonly5953/gemma2-9b-cpt-sahabatai-v1-instruct-GGUF"
LLM_MODEL_FILE = "gemma2-9b-cpt-sahabatai-v1-instruct.q4_k_m.gguf"
LLM_N_CTX = 8192            # prompt + jawaban, dalam token
MAX_OUTPUT_TOKENS = 4096    # pengaman; panjang jawaban sudah dibatasi di schema
api_call_count = 0

if not os.path.exists("/dev/nvidia0"):
    raise RuntimeError("Butuh GPU: Runtime -> Change runtime type -> T4 GPU, lalu jalankan ulang dari atas.")
if "llm" not in globals():   # dimuat sekali; kalau dimuat ulang, memori GPU habis
    # unduhan pertama ~5.8 GB; n_gpu_layers=-1 = semua layer di GPU
    llm = Llama.from_pretrained(repo_id=LLM_MODEL_REPO, filename=LLM_MODEL_FILE, n_ctx=LLM_N_CTX,
                                n_gpu_layers=-1, verbose=False)


def ask(prompt, schema):
    """Kirim prompt, jawaban dipaksa sesuai JSON schema lalu jadi objek. Gagal -> None.
    Setiap list/teks di schema wajib punya max_length: tanpa batas, model bisa menulis terus sampai terpotong."""
    global api_call_count
    api_call_count += 1
    try:
        out = llm.create_chat_completion(
            messages=[{"role": "user", "content": prompt}], temperature=0, max_tokens=MAX_OUTPUT_TOKENS,
            response_format={"type": "json_object", "schema": schema.model_json_schema()})
        return schema.model_validate_json(out["choices"][0]["message"]["content"])
    except Exception as e:   # biasanya prompt melebihi LLM_N_CTX
        print(f"Jawaban AI tidak valid: {str(e)[:200]}")
        return None

## 3. Settings form

In [41]:
PROJECT = "prabowo_vladivostok"  # @param {type:"string"}
TOPIC = "Kunjungan Presiden Prabowo ke Vladivostok, Rusia, dan pertemuannya dengan Vladimir Putin"  # @param {type:"string"}
SOURCES_TEXT = "https://docs.google.com/spreadsheets/d/1rVvruuhS569La9-lm0qNiIAUoQp-DnnF/edit?usp=drive_link&ouid=116825097454650626545&rtpof=true&sd=true"  # @param {type:"string"}
REGENERATE_TOPIC_CONFIG = True  # @param {type:"boolean"}

PROJECT = re.sub(r"[^a-z0-9_]+", "_", PROJECT.strip().lower()).strip("_")
TOPIC = TOPIC.strip()
SOURCES = [s for s in re.split(r"[,\s]+", SOURCES_TEXT) if s]
if not (PROJECT and TOPIC and SOURCES):
    raise ValueError("Isi PROJECT, TOPIC, dan SOURCES_TEXT terlebih dahulu.")
print(f"{PROJECT} | {TOPIC} | {len(SOURCES)} file")

prabowo_vladivostok | Kunjungan Presiden Prabowo ke Vladivostok, Rusia, dan pertemuannya dengan Vladimir Putin | 1 file


## 4. Configuration

In [ ]:
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive"

COL_HEADLINE = "Headline"
COL_MENTIONS = "Mentions"
COL_MEDIA = "Media"
COL_AUTHOR_ID = "Author"
COL_FOLLOWERS = "Followers"
COL_RETWEETED = "Retweeted"
COL_FAVOURITED = "Favourited"
EXPECTED_COLS = [c for c in EXPORT_COLUMNS if c != "Location"]

# bobot skor top author (total 1.0)
WEIGHT_POST_COUNT = 0.6
WEIGHT_ENGAGEMENT = 0.25
WEIGHT_FOLLOWERS = 0.15

BATCH_SIZE = 20              # post per request AI
TOP_N_AUTHORS = 10           # per tema
MAX_POSTS_PER_AUTHOR = 15    # post yang dibaca AI untuk sentimen
MIN_THEME_CONFIDENCE = 0.6   # di bawah ini tidak ikut ranking
DEDUP_PER_AUTHOR = False     # True = post kembar dari author berbeda tetap dihitung

TOPIC_CONFIG_PATH = f"{DRIVE_DIR}/topic_config_{PROJECT}.json"
CHECKPOINT_PATH = f"{DRIVE_DIR}/checkpoint_tema_{PROJECT}.pkl"
THEME_RESULT_PATH = f"{DRIVE_DIR}/df_clean_after_theme_{PROJECT}.pkl"
TOP_AUTHORS_PATH = f"{DRIVE_DIR}/kol_{PROJECT}.csv"
SUMMARY_PATH = f"{DRIVE_DIR}/kol_summary_{PROJECT}.csv"
PENDING_TEMA_PATH = f"{DRIVE_DIR}/pending_tema_data.jsonl"   # data latih tema, dipakai semua topik

## 5. Topic configuration (generated by the LLM)

In [ ]:
TOPIC_CONFIG_PROMPT = f"""You are a media analyst at a socio-political think tank, setting up monitoring of Indonesian online news and social media.

Topic: "{TOPIC}" (today: {date.today()})

The raw data is broader than this topic (e.g. every post that mentions its main figure). Posts without any keyword are dropped; the rest are classified into themes.

Return JSON with exactly these fields:
1. themes         - 2-4 themes that split the discussion by angle (e.g. event, policy, political reaction), not by person. Each has a short snake_case name and a description: "Masuk: <what it covers>. Contoh frasa: <8-15 typical phrases from news and social media, incl. slang>. Bukan: <similar things that belong to another theme>." Do not create a "lainnya" theme.
2. keywords       - 20-60 lowercase words or short phrases that almost only appear when this topic is discussed: other parties, places, events, institutions, abbreviations, common typos, hashtags without #. Matched as whole words, so prefer single words. Never the main figure's name alone, never general words used in other issues (e.g. "investasi", "kunjungan").
3. media_accounts - lowercase usernames (no @) of news outlets or official institution accounts tied to this topic. Exclude personal accounts, buzzers, and influencers. If unsure, leave it out; an empty list is fine.

Answer in Indonesian. Output only JSON."""


Short = Annotated[str, Field(max_length=40)]


class Theme(BaseModel):
    name: Short
    description: str = Field(max_length=800)


class TopicConfig(BaseModel):
    themes: list[Theme] = Field(min_length=2, max_length=4)
    keywords: list[Short] = Field(min_length=1, max_length=60)
    media_accounts: list[Short] = Field(max_length=30)


def check_config(cfg):
    """Rapikan config: nama tema snake_case, keyword huruf kecil, buang yang tidak valid."""
    themes = {re.sub(r"\W+", "_", k.strip().lower()).strip("_"): v for k, v in cfg["themes"].items()}
    themes.pop("lainnya", None)
    keywords = {k.strip().lower().lstrip("#") for k in cfg["keywords"]}
    keywords = sorted(k for k in keywords if len(k) >= 3)   # kata 1-2 huruf terlalu ambigu
    if not themes or not keywords:
        raise RuntimeError("Tema/keyword dari AI kosong. Jalankan ulang cell ini.")
    media = sorted({a.strip().lower().lstrip("@") for a in cfg.get("media_accounts", [])})
    return {"themes": themes, "keywords": keywords, "media_accounts": media}


topic_config = {}
if os.path.exists(TOPIC_CONFIG_PATH) and not REGENERATE_TOPIC_CONFIG:
    with open(TOPIC_CONFIG_PATH, encoding="utf-8") as f:
        topic_config = json.load(f)

if topic_config.get("_topic") != TOPIC:   # belum ada, diminta dibuat ulang, atau milik topik lain
    answer = ask(TOPIC_CONFIG_PROMPT, TopicConfig)
    if answer is None:
        raise RuntimeError("AI gagal membuat konfigurasi topik. Jalankan ulang cell ini.")
    topic_config = {"themes": {t.name: t.description for t in answer.themes},
                    "keywords": answer.keywords, "media_accounts": answer.media_accounts, "_topic": TOPIC}
    with open(TOPIC_CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(topic_config, f, ensure_ascii=False, indent=2)

topic_config = check_config(topic_config)
THEMES_DESC = {**topic_config["themes"], "lainnya": "di luar tema di atas"}
THEMES = list(THEMES_DESC)
RELEVANT_THEMES = THEMES[:-1]
KEYWORDS = sorted(topic_config["keywords"], key=len, reverse=True)   # frasa panjang dicocokkan lebih dulu
MEDIA_ACCOUNTS = set(topic_config["media_accounts"])
RE_KEYWORD = re.compile(r"(?<!\w)(?:" + "|".join(map(re.escape, KEYWORDS)) + r")(?!\w)")

print(json.dumps(topic_config, ensure_ascii=False, indent=2))

## 6. Load data

In [ ]:
def xlsx_url(link):
    """Link Google Sheets/Drive -> link download .xlsx."""
    match = re.search(r"/d/([\w-]+)", link)
    if not match:
        return link
    if "drive.google.com" in link:
        return f"https://drive.google.com/uc?export=download&id={match.group(1)}"
    return f"https://docs.google.com/spreadsheets/d/{match.group(1)}/export?format=xlsx"


def load_sheet(link):
    try:
        df = pd.read_excel(xlsx_url(link), header=1)
    except ValueError as e:
        raise ValueError(f"Gagal baca {link}. Pastikan akses file 'Anyone with the link'.") from e
    df.columns = df.columns.str.strip()
    return df[EXPECTED_COLS]


df_raw = pd.concat([load_sheet(s).assign(source_file=f"file_{i}") for i, s in enumerate(SOURCES, 1)],
                   ignore_index=True)
print(f"{len(df_raw)} baris")
display(df_raw.head())

## 7. Remove media accounts

In [ ]:
NUM_COLS = [COL_FOLLOWERS, COL_RETWEETED, COL_FAVOURITED]
df_raw[NUM_COLS] = df_raw[NUM_COLS].apply(pd.to_numeric, errors="coerce").fillna(0)

is_media = pd.Series([is_media_account(a, m, extra_accounts=MEDIA_ACCOUNTS)
                      for a, m in zip(df_raw[COL_AUTHOR_ID], df_raw[COL_MEDIA])], index=df_raw.index)
print(f"Buang {is_media.sum()} baris media, contoh akun:",
      df_raw.loc[is_media, COL_AUTHOR_ID].astype(str).unique()[:20].tolist())
df_raw = df_raw[~is_media].reset_index(drop=True)

## 8. Clean text

In [ ]:
# Retweet ditulis "RT <teks>", balasan ditulis "... [RE username]"
RE_RT = re.compile(r"^\s*RT\b\s*(?:@\w+\s*:\s*)?")
RE_REPLY = re.compile(r"\[RE\s+@?\w+\]", re.IGNORECASE)
RE_LONG = re.compile(r"(.)\1{2,}")   # "mantaaap" -> "mantaap"


def strip_noise(text):
    """Buang tanda RT/[RE], emoji jadi kata, huruf yang diulang-ulang dipendekin."""
    text = RE_REPLY.sub(" ", RE_RT.sub("", text))
    text = emoji.demojize(text, language="id").replace("_", " ").replace(":", " ")
    return RE_LONG.sub(r"\1\1", text)


def combine_text(headline, mentions):
    headline = "" if pd.isna(headline) else str(headline).strip()
    mentions = "" if pd.isna(mentions) else str(mentions).strip()
    if headline.lower() in ("", "nan", mentions.lower()):
        return mentions or headline
    return f"{headline}. {mentions}" if mentions else headline


df_raw["text_raw"] = [combine_text(h, m) for h, m in zip(df_raw[COL_HEADLINE], df_raw[COL_MENTIONS])]
df_raw["is_retweet"] = df_raw["text_raw"].str.match(RE_RT)
no_noise = df_raw["text_raw"].map(strip_noise)
df_raw["text_clean"] = no_noise.progress_apply(clean_for_bert)   # untuk filter, cek duplikat & keyword
df_raw["text_ai"] = no_noise.apply(clean_for_ner)                # untuk AI: huruf besar, slang & @akun tetap

df_clean = df_raw[df_raw["text_clean"].str.split().str.len() >= 3]

# buang duplikat; post asli didahulukan agar yang tersisa milik penulis aslinya
dup_key = df_clean["text_clean"].str.lower().str.replace(r"[^\w\s]", "", regex=True).str.split().str.join(" ")
dup_cols = ["dup_key", COL_AUTHOR_ID] if DEDUP_PER_AUTHOR else ["dup_key"]
is_dup = df_clean.assign(dup_key=dup_key).sort_values("is_retweet", kind="stable").duplicated(dup_cols).sort_index()
df_clean = df_clean[~is_dup].reset_index(drop=True)

print(f"{len(df_raw)} -> {len(df_clean)} baris setelah teks pendek & duplikat dibuang "
      f"({df_raw['is_retweet'].sum()} retweet)")
display(df_clean[["text_raw", "text_clean", "text_ai"]].head())

## 9. Assign themes (LLM)

In [ ]:
THEME_LIST = "\n".join(f"- {t}: {d}" for t, d in THEMES_DESC.items())


def theme_prompt(texts):
    numbered = "\n".join(f'{i}. "{t}"' for i, t in enumerate(texts, 1))
    return f"""You are a media analyst at a socio-political think tank, classifying Indonesian news and social media posts about: {TOPIC}.

Themes:
{THEME_LIST}

Posts:
{numbered}

Return JSON with one entry per post ({len(texts)} posts), each with these fields:
1. index         - the post number.
2. primary_theme - one theme name from the list. Use "lainnya" if the post is not directly about the topic (e.g. it only mentions the same person, country, or institution for another event).
3. confidence    - 0 to 1.

Output only JSON."""


class PostTheme(BaseModel):
    index: int
    primary_theme: Literal[tuple(THEMES)]
    confidence: float


def theme_answer(n):
    """Schema jawaban dengan tepat n post, supaya tidak ada post yang dilewati."""
    return create_model("ThemeAnswer", posts=(list[PostTheme], Field(min_length=n, max_length=n)))


def save_checkpoint(next_batch):
    pd.to_pickle({"run_id": run_id, "df": df_clean, "todo": todo, "next": next_batch, "tema_rows": tema_rows},
                 CHECKPOINT_PATH)


# checkpoint hanya dilanjutkan kalau topik, tema, dan datanya masih sama
run_id = [TOPIC, THEMES, int(pd.util.hash_pandas_object(df_clean["text_clean"], index=False).sum())]
ckpt = pd.read_pickle(CHECKPOINT_PATH) if os.path.exists(CHECKPOINT_PATH) else {}

if ckpt.get("run_id") == run_id:
    df_clean, todo, start, tema_rows = ckpt["df"], ckpt["todo"], ckpt["next"], ckpt["tema_rows"]
    print(f"Lanjut dari batch {start}")
else:
    df_clean[["theme", "theme_confidence"]] = None
    # post tanpa keyword langsung "lainnya", tidak dikirim ke AI
    text_lower = df_clean["text_clean"].str.lower()
    no_kw = ~text_lower.str.contains(RE_KEYWORD)
    df_clean.loc[no_kw, ["theme", "theme_confidence"]] = ["lainnya", 1.0]
    print(f"{no_kw.sum()} post tanpa keyword -> 'lainnya', {(~no_kw).sum()} post dikirim ke AI")
    print("Keyword paling sering:",
          Counter(k for t in text_lower[~no_kw] for k in set(RE_KEYWORD.findall(t))).most_common(10))
    if (~no_kw).mean() > 0.5:
        print("Lebih dari separuh post lolos; kemungkinan ada keyword yang terlalu umum (lihat di atas). "
              "Centang REGENERATE_TOPIC_CONFIG lalu jalankan ulang.")
    todo = df_clean.index[df_clean["theme"].isna()].tolist()
    start, tema_rows = 0, []

n_batches = -(-len(todo) // BATCH_SIZE)
for b in tqdm(range(start, n_batches), initial=start, total=n_batches, desc="Tema"):
    idx = todo[b * BATCH_SIZE:(b + 1) * BATCH_SIZE]
    texts = df_clean.loc[idx, "text_ai"].tolist()
    answer = ask(theme_prompt(texts), theme_answer(len(texts)))
    if answer is None:
        print(f"Batch {b}: {len(idx)} post tidak mendapat tema dari AI.")
    for i, p in zip(idx, answer.posts if answer else []):   # jumlah jawaban = jumlah post, urutannya sama
        df_clean.loc[i, ["theme", "theme_confidence"]] = [p.primary_theme, p.confidence]
        tema_rows.append({"text": df_clean.at[i, "text_clean"], "label": p.primary_theme, "confidence": p.confidence})

    if b % 10 == 9:   # kalau runtime putus, jalankan ulang cell ini untuk lanjut
        save_checkpoint(b + 1)

# data latih ditulis sekali di akhir agar tidak dobel saat melanjutkan checkpoint
with open(PENDING_TEMA_PATH, "a", encoding="utf-8") as f:
    for r in tema_rows:
        f.write(json.dumps({**r, "label_source": "llm", "llm_model": LLM_MODEL_FILE, "task": "tema",
                            "project": PROJECT, "run_date": str(date.today())}, ensure_ascii=False) + "\n")
if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)

n_missing = df_clean["theme"].isna().sum()
if n_missing:
    print(f"{n_missing} post belum punya tema, jadi hasil di bawah belum lengkap.")
display(df_clean["theme"].value_counts(dropna=False))
df_clean.to_pickle(THEME_RESULT_PATH)
print(f"Disimpan ke {THEME_RESULT_PATH}. Total request AI: {api_call_count}")

## 10. Top authors per theme

In [ ]:
relevant = df_clean["theme"].isin(RELEVANT_THEMES)
confident = df_clean["theme_confidence"].astype(float) >= MIN_THEME_CONFIDENCE
# retweet tidak dihitung karena bukan tulisan akun itu sendiri
df_posts = df_clean[relevant & confident & ~df_clean["is_retweet"]].copy()
df_posts["engagement"] = df_posts[COL_FAVOURITED] + df_posts[COL_RETWEETED]
print(f"{len(df_posts)} post asli ikut ranking. Dibuang: {(relevant & ~confident).sum()} tema kurang yakin, "
      f"{(relevant & confident & df_clean['is_retweet']).sum()} retweet")

authors = (df_posts.groupby(["theme", COL_AUTHOR_ID])
           .agg(jumlah_post=("engagement", "size"),
                total_engagement=("engagement", "sum"),
                followers=(COL_FOLLOWERS, "max"))
           .reset_index()
           .rename(columns={COL_AUTHOR_ID: "author_id"}))
metrics = authors[["jumlah_post", "total_engagement", "followers"]].astype(float)
# dibagi nilai terbesar di temanya, jadi skalanya 0-1
norm = metrics / metrics.groupby(authors["theme"]).transform("max").replace(0, 1)
authors["score"] = norm @ [WEIGHT_POST_COUNT, WEIGHT_ENGAGEMENT, WEIGHT_FOLLOWERS]

top_authors = (authors.sort_values(["theme", "score"], ascending=[True, False])
               .groupby("theme").head(TOP_N_AUTHORS)
               .reset_index(drop=True))
top_authors["rank"] = top_authors.groupby("theme").cumcount() + 1
top_authors.to_csv(TOP_AUTHORS_PATH, index=False)

for theme in RELEVANT_THEMES:
    group = top_authors[top_authors["theme"] == theme]
    print(f"\n=== {theme} ===")
    if group.empty:
        print("tidak ada author")
    else:
        display(group[["rank", "author_id", "jumlah_post", "total_engagement", "followers", "score"]])

## 11. Stance and summary per author (LLM)

In [ ]:
def sentiment_prompt(author_id, theme, posts):
    joined = "\n".join(f"- {p}" for p in posts)
    return f"""You are a media analyst at a socio-political think tank, profiling one account's stance.

These posts by "{author_id}" are about the theme "{theme.replace('_', ' ')}" of the topic "{TOPIC}":
{joined}

Return JSON with exactly these fields:
1. sentiment - the account's stance toward the topic, not its tone: positif (supports, praises, defends), negatif (criticises, rejects, mocks), netral (only informs, no clear stance), or kontroversi (support and criticism equally strong). Defending the main figure while insulting critics is positif.
2. summary   - 2-3 sentences on the account's main view, grounded in the posts.
3. reason    - one sentence explaining the label, grounded in the posts.

Answer in Indonesian. Output only JSON."""


class Sentiment(BaseModel):
    sentiment: Literal["positif", "negatif", "netral", "kontroversi"]
    summary: str = Field(max_length=600)
    reason: str = Field(max_length=300)


results = []
for _, a in tqdm(top_authors.iterrows(), total=len(top_authors), desc="Sentimen"):
    posts = (df_posts[(df_posts[COL_AUTHOR_ID] == a["author_id"]) & (df_posts["theme"] == a["theme"])]
             .nlargest(MAX_POSTS_PER_AUTHOR, "engagement")["text_ai"].tolist())
    answer = ask(sentiment_prompt(a["author_id"], a["theme"], posts), Sentiment)
    results.append(answer.model_dump() if answer else {"sentiment": None, "summary": "", "reason": ""})

df_summary = pd.concat([top_authors, pd.DataFrame(results, columns=["sentiment", "summary", "reason"])], axis=1)
df_summary.to_csv(SUMMARY_PATH, index=False)

n_failed = df_summary["sentiment"].isna().sum()
if n_failed:
    print(f"{n_failed} author gagal dinilai AI (sentiment kosong). Jalankan ulang cell ini.")
print(f"Disimpan ke {SUMMARY_PATH}. Total request AI: {api_call_count}")

pd.set_option("display.max_colwidth", None)
display(df_summary.round({"score": 2}))